In [31]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [32]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [33]:
target_files = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]
selected = [d for d in documents if d["filename"] in target_files]

In [34]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel

from evaluation_utils import llm_structured

In [35]:

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [36]:
class Questions(BaseModel):
    questions: list[str]

In [37]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [38]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [39]:

input_token_counts = []

for doc in selected:
    questions, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        json.dumps(doc),
        Questions,
    )
    # field name depends on provider: input_tokens (OpenAI Responses)
    # or prompt_tokens (Chat Completions)
    input_token_counts.append(usage.input_tokens)  # or usage.prompt_tokens

avg_input_tokens = sum(input_token_counts) / len(input_token_counts)
print(avg_input_tokens)

1353.0


In [40]:
import pandas as pd

ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")

In [41]:
ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [42]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [43]:
print(chunks[0].keys())

dict_keys(['start', 'content', 'filename'])


In [44]:
from minsearch import Index

# check what the text field is called:
print(chunks[0].keys())   # e.g. dict_keys(['filename', 'start', 'content'])

text_index = Index(
    text_fields=["content"],          # use whatever the content field is named
    keyword_fields=["filename"],    # keyed on filename
)
text_index.fit(chunks)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

dict_keys(['start', 'content', 'filename'])


In [45]:
q = ground_truth[0]["question"]
results = text_search(q)
print(results[0]["filename"])

01-agentic-rag/lessons/03-rag.md


In [46]:
from embedder import Embedder
model = Embedder()   # uses models/Xenova/all-MiniLM-L6-v2

In [47]:
chunks[0].keys()

dict_keys(['start', 'content', 'filename'])

In [48]:
import numpy as np
from minsearch import VectorSearch

# embed the text of each chunk (use the same field you used for text search)
chunk_texts = [c["content"] for c in chunks]           
X = np.array(model.encode_batch(chunk_texts))

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)                                 # X = vectors, chunks = payloads

In [49]:
def vector_search(query, num_results=5):
    query_vector = model.encode(query)
    return vindex.search(query_vector, num_results=num_results)

In [50]:
q = ground_truth[0]["question"]
results = vector_search(q)
print(results[0]["filename"])

01-agentic-rag/lessons/01-intro.md


In [51]:
from tqdm.auto import tqdm

def hit_rate(relevance_total):
    cnt = 0
    for line in relevance_total:
        if True in [x == 1 for x in line]:
            cnt += 1
    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance_total)

In [52]:
def compute_relevance(q, search_function):
    filename = q["filename"]                 # <- was q["document"] in the module
    results = search_function(q["question"])
    return [int(d["filename"] == filename) for d in results]   # <- compare filename

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance_total.append(compute_relevance(q, search_function))
    return relevance_total

In [53]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [54]:
evaluate(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [55]:
# precompute all query vectors in one batch
questions = [q["question"] for q in ground_truth]
question_vectors = model.encode_batch(questions)

def compute_relevance_precomputed(ground_truth, vectors):
    relevance_total = []
    for q, qv in zip(ground_truth, vectors):
        results = vindex.search(qv, num_results=5)
        relevance_total.append([int(d["filename"] == q["filename"]) for d in results])
    return relevance_total

rel = compute_relevance_precomputed(ground_truth, question_vectors)
print({"hit_rate": hit_rate(rel), "mrr": mrr(rel)})

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}


In [56]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [57]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [58]:
for k in [1, 50, 100, 200]:
    search_fn = lambda query, k=k: hybrid_search(query, k=k)   # bind k
    result = evaluate(ground_truth, search_fn)
    print(k, result["mrr"])

  0%|          | 0/360 [00:00<?, ?it/s]

1 0.6481944444444449


  0%|          | 0/360 [00:00<?, ?it/s]

50 0.637916666666667


  0%|          | 0/360 [00:00<?, ?it/s]

100 0.637916666666667


  0%|          | 0/360 [00:00<?, ?it/s]

200 0.637916666666667
